# UR5 equations of motion — three forward-dynamics pipelines

Textbook-style comparison of how minilink evaluates UR5 joint accelerations $\ddot q$ from

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + d(q,\dot q) + g(q) = \tau.
$$

Standalone notebook: only **minilink** (+ NumPy / SymPy / JAX). Helpers live in a collapsible cell below.

**§0** free-motion · **§1–4** math · **§5** one-shot · **§6** ABA vs RNEA (fast) · **§7** symbolic build · **§8** three-way speed · **§9** short EoM integration.

| Pipeline | Idea | minilink entry |
| --- | --- | --- |
| **RNEA–$H$** | RNEA bias + explicit inertia solve | `UR5Manipulator.forward_dynamics_rnea_h` |
| **ABA** | Articulated-body algorithm ($O(n)$ spatial) | `UR5Manipulator.forward_dynamics` (catalog default) |
| **Symbolic Lagrange** | Derive $H,C,g$ once; lambdify to JAX | `minilink.symbolic` → `to_minilink(backend="jax")` |

**Inverse dynamics:** catalog default is spatial **RNEA** (`inverse_dynamics`); matrix form is `inverse_dynamics_matrix`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

# Local conda: minilink already installed. Colab: clone + path + meshcat.
import importlib.util
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

_OPTIMIZER_METHOD = (
    "ipopt" if importlib.util.find_spec("cyipopt") is not None else "scipy_slsqp"
)


from minilink.dynamics.catalog.manipulators.ur5 import UR5Manipulator
from minilink.simulation.simulator import Simulator


<details>
<summary><b>Notebook helpers</b> — run once (click to expand / read)</summary>

Self-contained utilities:

- **Symbolic UR5** — DH chain → Lagrange derive → one JAX `to_minilink` export (cached); stage prints + ETA.
- **Parity / speed** — `parity_check` (`max |Δq̈|`) and `speed_compare` (`jax.jit` + warm-up excluded); durations as µs / ms / s.
- **EoM integration** — short RK4 for RNEA–$H$ vs ABA; time *compile* / *JIT* / *integrate*.

Defaults: `5` samples, `5` timing repeats.

</details>


In [ ]:
# --- Notebook helpers (self-contained) ---
import time
from dataclasses import dataclass, field

DEFAULT_SEED = 0
DEFAULT_N_SAMPLES = 5  # small reproducible batch
DEFAULT_N_TIMING = 5
DEFAULT_PARITY_TOL = 1e-6
DEFAULT_EXPORT_ETA_S = 75.0  # prior for JAX export ETA after derive
DEFAULT_INTEGRATION_TF = 5.0
DEFAULT_INTEGRATION_DT = 0.05
DEFAULT_INTEGRATION_BACKEND = "jax"
DEFAULT_INTEGRATION_SOLVER = "rk4_fixedsteps"

_SYM_SYS = None
_PARAM_MAP = None
_PLANTS = {}


def catalog_params_no_damping():
    """Match symbolic Lagrange (no viscous d) to catalog spatial params."""
    params = dict(UR5Manipulator().params)
    params["damping"] = np.zeros(6)
    return params


def format_duration(seconds):
    """Human wall time: µs / ms / s (never scientific ms)."""
    s = float(seconds)
    if s < 0:
        s = abs(s)
    if s < 1e-3:
        return f"{1e6 * s:.0f} µs"
    if s < 1.0:
        return f"{1e3 * s:.1f} ms"
    return f"{s:.1f} s"


def format_error(err):
    """Parity error for print / tables."""
    e = float(err)
    if e == 0.0:
        return "0"
    return f"{e:.2e}"


def derive_symbolic_ur5(*, progress=True):
    """Lagrange derive once; cache SymPy H, C, g."""
    global _SYM_SYS, _PARAM_MAP
    if _SYM_SYS is not None:
        if progress:
            print("Symbolic derive: using cache.")
        return _SYM_SYS

    from minilink.symbolic.mechanics.model import MechanicalModel

    catalog = UR5Manipulator()
    params = catalog.params
    model = MechanicalModel("UR5Symbolic")
    coords = model.coordinates("q1 q2 q3 q4 q5 q6")
    g_sym = model.parameters("g")

    dh_table, link_properties = [], []
    for i in range(catalog.dof):
        inertia = params["inertia"][i]
        dh_table.append(
            {
                "theta": coords[i],
                "d": float(params["d"][i]),
                "a": float(params["a"][i]),
                "alpha": float(params["alpha"][i]),
            }
        )
        link_properties.append(
            {
                "mass": float(params["mass"][i]),
                "inertia": {
                    "Ixx": float(inertia[0, 0]),
                    "Iyy": float(inertia[1, 1]),
                    "Izz": float(inertia[2, 2]),
                },
                "com_offset": {
                    "x": float(params["com"][i, 0]),
                    "y": float(params["com"][i, 1]),
                    "z": float(params["com"][i, 2]),
                },
            }
        )

    model.add_dh_chain(dh_table, link_properties)
    model.add_gravity(-g_sym * model.N.z)

    if progress:
        print("Symbolic UR5: derive Lagrange (often ~1–3 min, simplify=False)...")
    t0 = time.perf_counter()
    sym_sys = model.derive(method="lagrange", simplify=False)
    derive_s = time.perf_counter() - t0
    if progress:
        eta_export = DEFAULT_EXPORT_ETA_S
        print(
            f"  derive done in {format_duration(derive_s)} — "
            f"next: JAX export (~{format_duration(eta_export)} prior)..."
        )

    _SYM_SYS = sym_sys
    _PARAM_MAP = {g_sym: float(params["gravity"])}
    return sym_sys


def build_symbolic_ur5(*, backend=DEFAULT_INTEGRATION_BACKEND, progress=True):
    """One SymPy → minilink export (default JAX). Cached per backend."""
    global _PLANTS
    if backend in _PLANTS:
        if progress:
            print(f"Symbolic export ({backend!r}): using cache.")
        return _PLANTS[backend]

    t_total0 = time.perf_counter()
    sym_sys = derive_symbolic_ur5(progress=progress)
    if progress:
        print(f"Exporting / lambdifying to minilink (backend={backend!r})...")
    t0 = time.perf_counter()
    plant = sym_sys.to_minilink(parameters=_PARAM_MAP, backend=backend)
    export_s = time.perf_counter() - t0
    total_s = time.perf_counter() - t_total0
    if progress:
        print(
            f"  export done in {format_duration(export_s)} "
            f"(total {format_duration(total_s)})"
        )
    _PLANTS[backend] = plant
    return plant


def _entry_term_count(expr):
    import sympy as sp

    if expr == 0:
        return 0
    return len(sp.Add.make_args(expr))


def symbolic_matrix_complexity(matrix):
    """Top-level term counts + count_ops for a SymPy matrix."""
    import sympy as sp

    M = sp.Matrix(matrix)
    rows, cols = M.shape
    term_counts = np.zeros((rows, cols), dtype=int)
    op_counts = np.zeros((rows, cols), dtype=int)
    for i in range(rows):
        for j in range(cols):
            term_counts[i, j] = _entry_term_count(M[i, j])
            op_counts[i, j] = int(sp.count_ops(M[i, j]))
    i_max, j_max = np.unravel_index(int(np.argmax(op_counts)), op_counts.shape)
    return {
        "shape": (rows, cols),
        "term_counts": term_counts,
        "op_counts": op_counts,
        "total_terms": int(term_counts.sum()),
        "max_terms": int(term_counts.max()),
        "mean_terms": float(term_counts.mean()),
        "total_ops": int(op_counts.sum()),
        "max_ops": int(op_counts.max()),
        "densest_entry": (int(i_max), int(j_max)),
        "n_zero": int(np.count_nonzero(term_counts == 0)),
    }


def format_complexity_table(stats):
    counts = stats["term_counts"]
    rows, cols = counts.shape
    header = "|  | " + " | ".join(f"j={j + 1}" for j in range(cols)) + " |"
    sep = "| --- | " + " | ".join("---" for _ in range(cols)) + " |"
    lines = [header, sep]
    for i in range(rows):
        cells = " | ".join(str(int(counts[i, j])) for j in range(cols))
        lines.append(f"| i={i + 1} | {cells} |")
    return "\n".join(lines)


def build_evaluation_batch(seed=DEFAULT_SEED, n_samples=DEFAULT_N_SAMPLES):
    """A few random (q, v, u) samples."""
    rng = np.random.default_rng(seed)
    configs = []
    for _ in range(n_samples):
        configs.append(
            (
                rng.uniform(-1.0, 1.0, 6),
                rng.uniform(-1.0, 1.0, 6),
                rng.uniform(-5.0, 5.0, 6),
            )
        )
    return configs


def forward_dynamics_rnea_h(arm, q, v, u, params):
    return arm.forward_dynamics_rnea_h(q, v, u, params=params)


def forward_dynamics_aba(arm, q, v, u, params):
    return arm.forward_dynamics(q, v, u, params=params)


def forward_dynamics_symbolic(plant, q, v, u):
    return plant.forward_dynamics(q, v, u)


def _block_until_ready(out):
    if hasattr(out, "block_until_ready"):
        out.block_until_ready()
    return out


def _jitted_forward_dynamics(func):
    """``jax.jit`` a ``(q, v, u) -> q̈`` callable (params closed over)."""
    import jax
    from minilink.core.backends import configure_jax

    configure_jax(enable_x64=True)
    return jax.jit(func)


def make_jitted_fd_catalog(arm, params):
    """JIT RNEA–H and ABA callables + JAX configs factory."""
    import jax.numpy as jnp

    fd_rnea = _jitted_forward_dynamics(
        lambda q, v, u: forward_dynamics_rnea_h(arm, q, v, u, params)
    )
    fd_aba = _jitted_forward_dynamics(
        lambda q, v, u: forward_dynamics_aba(arm, q, v, u, params)
    )

    def as_jax(configs):
        return [(jnp.asarray(q), jnp.asarray(v), jnp.asarray(u)) for q, v, u in configs]

    return fd_rnea, fd_aba, as_jax


def make_jitted_fd_symbolic(plant):
    return _jitted_forward_dynamics(
        lambda q, v, u: forward_dynamics_symbolic(plant, q, v, u)
    )


def _log(msg):
    """Print immediately (Jupyter often buffers without flush)."""
    print(msg, flush=True)


def parity_check(
    ref_fn, other_fn, configs, label, *, tol=DEFAULT_PARITY_TOL, progress=True
):
    """Print max |Δq̈| vs reference; return error and PASS/FAIL."""
    n = len(configs)
    if progress:
        _log(f"Parity {label} vs RNEA–H on {n} samples...")
    errs = []
    t_run0 = time.perf_counter()
    for i, (q, v, u) in enumerate(configs):
        if progress:
            _log(f"  {label}: sample {i + 1}/{n} (running)...")
        t0 = time.perf_counter()
        qdd_ref = np.asarray(ref_fn(q, v, u))
        qdd_other = np.asarray(other_fn(q, v, u))
        errs.append(float(np.max(np.abs(qdd_ref - qdd_other))))
        if progress:
            done = i + 1
            elapsed = time.perf_counter() - t_run0
            eta = elapsed / done * (n - done)
            _log(
                f"  {label}: sample {done}/{n} done in "
                f"{format_duration(time.perf_counter() - t0)}  "
                f"ETA {format_duration(eta)}"
            )
    max_err = float(np.max(errs))
    ok = max_err < tol
    status = "PASS" if ok else "FAIL"
    _log(
        f"Parity {label} vs RNEA–H: max |Δq̈| = {format_error(max_err)}  [{status}]"
        f"  (tol={format_error(tol)})"
    )
    return max_err, ok


def _benchmark_median(func, configs, n_repeat, *, label=None, progress=False):
    """Median wall time of ``func(q, v, u)`` (syncs JAX)."""
    times = []
    n_cfg = len(configs)
    t_run0 = time.perf_counter()
    for k in range(n_repeat):
        q, v, u = configs[k % n_cfg]
        if progress and label is not None:
            _log(f"  {label}: timing {k + 1}/{n_repeat} (running)...")
        t0 = time.perf_counter()
        _block_until_ready(func(q, v, u))
        times.append(time.perf_counter() - t0)
        if progress and label is not None:
            done = k + 1
            elapsed = time.perf_counter() - t_run0
            eta = elapsed / done * (n_repeat - done)
            _log(
                f"  {label}: {done}/{n_repeat}  "
                f"last={format_duration(times[-1])}  "
                f"ETA {format_duration(eta)}"
            )
    return float(np.median(times))


def speed_compare(named_fns, configs, *, n_timing=DEFAULT_N_TIMING, progress=True):
    """Warm JIT once each, then median call times. Returns {name: seconds}."""
    q0, v0, u0 = configs[0]
    if progress:
        _log(f"JIT warm-up ({len(named_fns)} pipelines)...")
    for name, fn in named_fns:
        if progress:
            hint = ""
            if name.lower().startswith("sym"):
                hint = " — first call compiles XLA, often minutes"
            _log(f"  {name}: warm-up starting{hint}...")
        t0 = time.perf_counter()
        _block_until_ready(fn(q0, v0, u0))
        if progress:
            _log(f"  {name}: warm-up done in {format_duration(time.perf_counter() - t0)}")

    if progress:
        _log(f"Timing median of {n_timing} calls (after warm-up)...")
    timing_s = {}
    for name, fn in named_fns:
        # Always show per-call progress when requested (Symbolic can be minutes/call).
        show = progress
        if progress:
            _log(f"Timing {name} ({n_timing} calls)...")
        timing_s[name] = _benchmark_median(
            fn, configs, n_timing, label=name, progress=show
        )
        if progress:
            _log(f"  {name}: {format_duration(timing_s[name])} / call (median)")
    return timing_s


def speed_table_markdown(timing_s, *, ref_name="RNEA-H"):
    """Markdown table: method | time/call | vs ref."""
    ref = timing_s.get(ref_name)
    rows = ["| method | time / call | vs RNEA–H |", "| --- | --- | --- |"]
    for name, t in timing_s.items():
        if ref is None or name == ref_name:
            rel = "1.00× (ref)" if name == ref_name else "—"
        else:
            rel = f"{ref / max(t, 1e-18):.2f}× faster" if t < ref else (
                f"{t / max(ref, 1e-18):.2f}× slower"
            )
        rows.append(f"| {name} | {format_duration(t)} | {rel} |")
    return "\n".join(rows)


class UR5ManipulatorRNEA(UR5Manipulator):
    """Same plant, but f uses RNEA–H forward dynamics (integration reference)."""

    def forward_dynamics(self, q, v, u, t=0.0, params=None):
        return self.forward_dynamics_rnea_h(q, v, u, t, params)


@dataclass
class IntegrationTiming:
    method: str
    compile_s: float
    jit_warmup_s: float
    integrate_s: float
    backend: str = DEFAULT_INTEGRATION_BACKEND


@dataclass
class IntegrationComparisonResult:
    backend: str
    tf: float
    dt: float
    n_steps: int
    timings: list = field(default_factory=list)
    max_state_error: dict = field(default_factory=dict)


def timed_integration(sys, *, method, compile_backend, tf, dt, solver, progress=True):
    """compile → first solve (JIT) → second solve (EoM integration only)."""
    if compile_backend == "jax":
        from minilink.core.backends import configure_jax

        configure_jax(enable_x64=True)

    if progress:
        print(f"[{method}] compile Simulator ({compile_backend})...")
    kwargs = dict(x0=sys.x0, t0=0.0, tf=tf, dt=dt, solver=solver, verbose=False)
    t0 = time.perf_counter()
    sim = Simulator(sys, compile_backend=compile_backend, **kwargs)
    compile_s = time.perf_counter() - t0
    if progress:
        print(f"  compile: {format_duration(compile_s)}")

    if progress:
        print(f"[{method}] JIT warm-up (first solve)...")
    t0 = time.perf_counter()
    sim.solve()
    jit_warmup_s = time.perf_counter() - t0
    if progress:
        print(f"  warm-up: {format_duration(jit_warmup_s)}")

    if progress:
        print(f"[{method}] integrate (second solve)...")
    t0 = time.perf_counter()
    traj = sim.solve()
    integrate_s = time.perf_counter() - t0
    if progress:
        print(f"  integrate: {format_duration(integrate_s)}")

    return traj, IntegrationTiming(
        method=method,
        compile_s=compile_s,
        jit_warmup_s=jit_warmup_s,
        integrate_s=integrate_s,
        backend=compile_backend,
    )


def eom_integration_comparison(
    params,
    *,
    symbolic_plant=None,
    compile_backend=DEFAULT_INTEGRATION_BACKEND,
    tf=DEFAULT_INTEGRATION_TF,
    dt=DEFAULT_INTEGRATION_DT,
    solver=DEFAULT_INTEGRATION_SOLVER,
    progress=True,
):
    """Integrate RNEA–H / ABA / optional symbolic on a short EoM grid."""
    q0 = np.array([0.0, -np.pi / 2 + 0.2, 0.0, -np.pi / 2, 0.0, 0.0])
    v0 = np.array([0.2, -0.1, 0.15, -0.05, 0.08, -0.03])

    def _prep(plant):
        plant.params = dict(params)
        plant.x0 = plant.q2x(q0, v0)
        plant.inputs["u"].nominal_value = np.zeros(plant.m)
        return plant

    result = IntegrationComparisonResult(
        backend=compile_backend,
        tf=tf,
        dt=dt,
        n_steps=int(round(tf / dt)) + 1,
    )

    ref_traj, ref_timing = timed_integration(
        _prep(UR5ManipulatorRNEA()),
        method="RNEA-H",
        compile_backend=compile_backend,
        tf=tf,
        dt=dt,
        solver=solver,
        progress=progress,
    )
    result.timings.append(ref_timing)

    aba_traj, aba_timing = timed_integration(
        _prep(UR5Manipulator()),
        method="ABA",
        compile_backend=compile_backend,
        tf=tf,
        dt=dt,
        solver=solver,
        progress=progress,
    )
    result.timings.append(aba_timing)
    result.max_state_error["ABA"] = float(np.max(np.abs(ref_traj.x - aba_traj.x)))

    if symbolic_plant is not None:
        sym_traj, sym_timing = timed_integration(
            _prep(symbolic_plant),
            method="Symbolic",
            compile_backend=compile_backend,
            tf=tf,
            dt=dt,
            solver=solver,
            progress=progress,
        )
        result.timings.append(sym_timing)
        result.max_state_error["Symbolic"] = float(
            np.max(np.abs(ref_traj.x - sym_traj.x))
        )

    return result, ref_traj


def rows_to_markdown(rows, columns, *, duration_cols=()):
    """Markdown table; duration_cols formatted with format_duration."""
    duration_cols = set(duration_cols)

    def cell(row, c):
        v = row[c]
        if c in duration_cols:
            return format_duration(v)
        if c.endswith("_err") or c.startswith("max_abs"):
            return format_error(v) if isinstance(v, float) else str(v)
        if isinstance(v, float):
            if v == 0.0:
                return "0"
            if abs(v) < 0.01 or abs(v) >= 100:
                return format_error(v)
            return f"{v:.2f}"
        return str(v)

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(cell(row, c) for c in columns) + " |" for row in rows]
    return chr(10).join([header, sep, *body])


def integration_timing_rows(result):
    return [
        {
            "method": t.method,
            "compile": t.compile_s,
            "jit_warmup": t.jit_warmup_s,
            "integrate": t.integrate_s,
            "backend": t.backend,
        }
        for t in result.timings
    ]


def integration_error_rows(result):
    return [{"method": m, "max_abs_dx": e} for m, e in result.max_state_error.items()]


## 0. Free-motion showcase

Catalog UR5 under gravity with **zero joint friction and no torque**. Meshcat HTML animation inline.


In [ ]:
demo = UR5Manipulator()
demo.params = catalog_params_no_damping()
q0 = np.array([0.0, -np.pi / 2 + 0.2, 0.0, -np.pi / 2, 0.0, 0.0])
v0 = np.array([0.25, -0.15, 0.2, -0.05, 0.1, -0.05])
demo.x0 = demo.q2x(q0, v0)
demo.compute_forced(
    lambda t: np.zeros(demo.m),
    tf=1.5,
    n_steps=45,
    compile_backend="jax",
    verbose=True,
)



In [ ]:
demo.animate(renderer="meshcat", is_3d=True, html=True)

## 1. Manipulator equation of motion

Consider a serial $n$-joint manipulator in generalized coordinates $q \in \mathbb{R}^n$. The standard second-order model is

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + d(q,\dot q) + g(q) = \tau,
$$

where $H(q) \in \mathbb{R}^{n \times n}$ is the symmetric positive-definite inertia matrix, $C(q,\dot q)\dot q$ collects Coriolis and centrifugal terms, $g(q)$ is gravity, $d$ is dissipation, and $\tau$ is applied joint torque.

Define the **bias force** (everything that is not inertia times acceleration):

$$
b(q,\dot q) = C(q,\dot q)\,\dot q + g(q).
$$

**Forward dynamics** solves for the generalized acceleration $\ddot q$ given $(q,\dot q,\tau)$:

$$
\ddot q = H(q)^{-1}\bigl(\tau - b(q,\dot q) - d(q,\dot q)\bigr).
$$

For simulation, minilink stacks state $x = [q;\dot q] \in \mathbb{R}^{2n}$ so that

$$
\dot x = \begin{bmatrix} \dot q \\ \ddot q \end{bmatrix} = f(x,u) = \begin{bmatrix} \dot q \\ \text{FD}(q,\dot q,u) \end{bmatrix}.
$$

On the UR5 catalog plant, **`H`**, **`C`**, and **`g`** are built from spatial RNEA; the default **`forward_dynamics`** uses the Articulated-Body Algorithm (ABA). Below we **disable viscous damping** ($d=0$) so the symbolic Lagrange export matches the spatial parameters.


## 2. Pipeline A — Recursive Newton–Euler + explicit $H$ (RNEA–$H$)

### 2.1 Spatial vectors

Each link $i$ carries a 6-vector spatial velocity $v_i = \begin{bmatrix} \omega_i \\ v_i \end{bmatrix}$ and spatial force $f_i$. The **spatial inertia** $I_i$ maps acceleration to force. The **motion cross** operator $\mathrm{crm}(v)$ and its dual $\mathrm{crm}(v)^\top$ appear in the force balance

$$
f_i = I_i a_i - \mathrm{crm}(v_i)^\top I_i v_i.
$$

### 2.2 Inverse dynamics (one tree pass each way)

Given $(q,\dot q,\ddot q)$, RNEA computes $\tau$ in two sweeps:

**Outward** (base $\to$ tip): for each joint $i$,

$$
v_i = {}^{i}\!X_{i-1}\, v_{i-1} + S_i\,\dot q_i, \qquad
a_i = {}^{i}\!X_{i-1}\, a_{i-1} + S_i\,\ddot q_i + \mathrm{crm}(v_i)\,S_i\,\dot q_i,
$$

then accumulate $f_i$ from $I_i$, $a_i$, and $v_i$.

**Inward** (tip $\to$ base): project forces onto joints,

$$
\tau_i = S_i^\top f_i, \qquad f_{i-1} \mathrel{+}= {}^{i}\!X_{i-1}^\top f_i.
$$

We write $\tau = \mathrm{RNEA}(q,\dot q,\ddot q)$.

### 2.3 Building $H$ and forward dynamics

The bias force at zero acceleration is

$$
b(q,\dot q) = \mathrm{RNEA}(q,\dot q, 0).
$$

Each column of the inertia matrix is one inverse-dynamics call with a unit joint acceleration:

$$
H_{:,j}(q) = \mathrm{RNEA}(q, 0, e_j), \qquad j = 1,\ldots,n,
$$

followed by symmetrization $H \leftarrow \tfrac12(H + H^\top)$. Forward dynamics is the linear solve

$$
\ddot q = H^{-1}(\tau - b - d).
$$

**Complexity:** one bias pass $O(n)$, $n$ columns $O(n^2)$, solve $O(n^3)$ — for UR5 ($n=6$) the solve is negligible; forming $H$ dominates.

**minilink:** `UR5Manipulator.forward_dynamics_rnea_h`, and `H` / `g` / `C` via the same spatial RNEA stack.


## 3. Pipeline B — Articulated Body Algorithm (ABA)

ABA computes the **same** $\ddot q$ as RNEA–$H$ but never assembles $H(q)$. It maintains articulated-body inertias $I_i^A$ and bias forces $p_{A,i}$ while propagating along the kinematic tree.

### 3.1 Pass 1 — outward (velocities and bias)

For each link $i$, with joint motion subspace $S_i$ and parent transform ${}^{i}\!X_{i-1}$:

$$
v_i = {}^{i}\!X_{i-1}\, v_{i-1} + S_i\,\dot q_i, \qquad
c_i = \mathrm{crm}(v_i)\,S_i\,\dot q_i,
$$

$$
p_{A,i} = -\mathrm{crm}(v_i)^\top I_i v_i.
$$

### 3.2 Pass 2 — inward (articulated inertia)

Initialize $I_i^A = I_i$. From tip to base, for each $i$:

$$
U_i = I_i^A S_i, \qquad d_i = S_i^\top U_i, \qquad u_i = \tau_i - S_i^\top p_{A,i},
$$

$$
I_i^A \leftarrow I_i^A - \frac{U_i U_i^\top}{d_i}, \qquad
p_{A,i-1} \mathrel{+}= {}^{i}\!X_{i-1}^\top\!\left(p_{A,i} + I_i^A c_i + \frac{U_i u_i}{d_i}\right),
$$

with the articulated inertia $I_i^A$ propagated to the parent before processing the next link.

### 3.3 Pass 3 — outward (accelerations)

Starting from the base spatial acceleration $a_0$ (gravity), for each $i$:

$$
a_i = {}^{i}\!X_{i-1}\, a_{i-1} + c_i, \qquad
\ddot q_i = \frac{u_i - U_i^\top a_i}{d_i}, \qquad
a_i \leftarrow a_i + S_i\,\ddot q_i.
$$

**Complexity:** each pass is $O(n)$, so **$O(n)$ per forward-dynamics call**.

**minilink:** `UR5Manipulator.forward_dynamics` (catalog default for simulation and `f`).


## 4. Pipeline C — Symbolic Lagrange derivation

### 4.1 Energies

Build a DH chain in SymPy. With kinetic energy $T(q,\dot q)$ and potential $V(q)$,

$$
L(q,\dot q) = T(q,\dot q) - V(q).
$$

For rigid links, $T = \sum_i \tfrac12 v_{c,i}^\top M_i v_{c,i} + \tfrac12 \omega_i^\top I_i \omega_i$ expressed in joint coordinates.

### 4.2 Euler–Lagrange equations

$$
\frac{d}{dt}\frac{\partial L}{\partial \dot q} - \frac{\partial L}{\partial q} = \tau.
$$

Expanding the $\dot q$-dependent terms yields the standard manipulator form

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + g(q) = \tau,
$$

where

$$
H_{ij} = \frac{\partial^2 T}{\partial \dot q_i \partial \dot q_j}, \qquad
g_i = \frac{\partial V}{\partial q_i},
$$

and the Coriolis matrix $C$ follows from Christoffel symbols of $H(q)$ (or equivalent Kane/Lagrange bookkeeping).

### 4.3 Export and evaluation

SymPy derives $H(q)$, $C(q,\dot q)$, and $g(q)$ symbolically. `to_minilink(backend="jax")` **lambdifies** them once into JAX callables; each forward-dynamics step evaluates $H$ and solves

$$
\ddot q = H^{-1}(\tau - C\dot q - g - d),
$$

so runtime cost is similar to RNEA–$H$, while the **upfront** derive + export cost is large (minutes for UR5).

**minilink:** `minilink.symbolic.mechanics` → `build_symbolic_ur5()` (helpers cell above).


### 4.4 Complexity summary

| Pipeline | Dominant per-step cost | Forms $H(q)$? |
| --- | --- | --- |
| RNEA–$H$ | $O(n^2)$ from $n$ RNEA columns | yes |
| ABA | $O(n)$ three tree passes | no |
| Symbolic Lagrange (numeric) | $O(n^2)$ evaluate $H$ + $O(n^3)$ solve | yes |

All three target the same $\ddot q$ when the underlying model data match; they differ in **algorithm** and **when** work is paid (symbolic: mostly upfront).


## 5. Hands-on — one configuration

Call ABA and RNEA–$H$ on the same $(q, \dot q, \tau)$ before any batch or symbolic work.


In [ ]:
arm = UR5Manipulator()
params = catalog_params_no_damping()

q = np.array([0.1, -0.5, 0.2, -1.0, 0.3, 0.0])
v = np.array([0.5, -0.3, 0.2, 0.1, -0.4, 0.2])
u = np.zeros(6)

qdd_rnea = forward_dynamics_rnea_h(arm, q, v, u, params)
qdd_aba = forward_dynamics_aba(arm, q, v, u, params)

print("RNEA–H qdd:", np.array2string(qdd_rnea, precision=4, suppress_small=True))
print("ABA    qdd:", np.array2string(qdd_aba, precision=4, suppress_small=True))
print("max |Δqdd| (ABA vs RNEA–H):", format_error(np.max(np.abs(qdd_rnea - qdd_aba))))


## 6. Fast catalog compare — ABA vs RNEA–$H$

Small random batch (seed 0). **Parity** then **JIT speed** for catalog pipelines only — should finish in seconds.


In [ ]:
SEED = DEFAULT_SEED
N_SAMPLES = DEFAULT_N_SAMPLES
N_TIMING = DEFAULT_N_TIMING

configs = build_evaluation_batch(seed=SEED, n_samples=N_SAMPLES)
fd_rnea, fd_aba, as_jax = make_jitted_fd_catalog(arm, params)
configs_j = as_jax(configs)

print(f"Batch: {len(configs_j)} samples (seed={SEED})")
parity_check(fd_rnea, fd_aba, configs_j, "ABA")

timing_catalog = speed_compare(
    [("RNEA-H", fd_rnea), ("ABA", fd_aba)],
    configs_j,
    n_timing=N_TIMING,
    progress=True,
)
display(Markdown("### Speed (catalog only)"))
display(Markdown(speed_table_markdown(timing_catalog)))


## 7. Symbolic pipeline — derive + JAX export

Copy catalog DH / mass / inertia into SymPy, derive Lagrange $H,C,g$ (`simplify=False`), then **lambdify once** to a JAX plant. Reused in §8 — do not export again.

This cell is slow (often a few minutes). Progress lines print derive / export timing and an ETA prior for export. Set `SKIP_SYMBOLIC = True` to skip.


In [ ]:
SKIP_SYMBOLIC = False  # set True to skip the long symbolic path

symbolic_plant = None
if not SKIP_SYMBOLIC:
    try:
        symbolic_plant = build_symbolic_ur5(
            backend=DEFAULT_INTEGRATION_BACKEND, progress=True
        )
        qdd_sym = forward_dynamics_symbolic(symbolic_plant, q, v, u)
        print("Symbolic qdd:", np.array2string(qdd_sym, precision=4, suppress_small=True))
        print(
            "max |Δqdd| (Symbolic vs RNEA–H):",
            format_error(np.max(np.abs(qdd_rnea - qdd_sym))),
        )
    except ImportError:
        print("SymPy not installed — pip install minilink[symbolic]")
else:
    print("Skipping symbolic pipeline.")


### Symbolic inertia $H(q)$ — size and complexity

Before lambdify, $H$ is an explicit SymPy matrix. Term counts = top-level summands; ops = `sympy.count_ops`.


In [ ]:
if symbolic_plant is None:
    print("Symbolic plant not built — skip complexity cell.")
else:
    print("Computing SymPy matrix complexity (H, C, g)...")
    t0 = time.perf_counter()
    sym_sys = derive_symbolic_ur5(progress=False)
    H_stats = symbolic_matrix_complexity(sym_sys.H)
    C_stats = symbolic_matrix_complexity(sym_sys.C)
    g_stats = symbolic_matrix_complexity(sym_sys.g)
    print(f"  complexity done in {format_duration(time.perf_counter() - t0)}")

    rows = []
    for name, st in (("H(q)", H_stats), ("C(q, q̇)", C_stats), ("g(q)", g_stats)):
        rows.append(
            {
                "matrix": name,
                "shape": f"{st['shape'][0]}×{st['shape'][1]}",
                "total_terms": st["total_terms"],
                "max_terms_entry": st["max_terms"],
                "mean_terms": f"{st['mean_terms']:.1f}",
                "total_ops": st["total_ops"],
                "zeros": st["n_zero"],
            }
        )
    cols = [
        "matrix",
        "shape",
        "total_terms",
        "max_terms_entry",
        "mean_terms",
        "total_ops",
        "zeros",
    ]
    display(Markdown("#### Aggregate complexity"))
    display(Markdown(rows_to_markdown(rows, cols)))
    display(Markdown("#### Top-level term counts in $H_{ij}$"))
    display(Markdown(format_complexity_table(H_stats)))

    i_d, j_d = H_stats["densest_entry"]
    entry_str = str(sym_sys.H[i_d, j_d])
    display(
        Markdown(
            f"Most-ops entry **$H_{{{i_d + 1},{j_d + 1}}}$**: "
            f"{int(H_stats['term_counts'][i_d, j_d])} terms, "
            f"{int(H_stats['op_counts'][i_d, j_d])} ops, "
            f"string length {len(entry_str):,}."
        )
    )
    print(entry_str[:900] + (" ..." if len(entry_str) > 900 else ""))


## 8. Three-way compare — parity + speed

Same JIT’d $\ddot q$ callables for accuracy and timing. Warm-up excluded from the table. Symbolic row omitted if §7 was skipped.


In [ ]:
# Reuse catalog JIT callables from §6; add Symbolic when available.
named = [("RNEA-H", fd_rnea), ("ABA", fd_aba)]
fd_sym = None
if symbolic_plant is not None:
    print("Wrapping Symbolic FD in jax.jit (compile happens on first call)...", flush=True)
    fd_sym = make_jitted_fd_symbolic(symbolic_plant)
    named.append(("Symbolic", fd_sym))
    q0, v0, u0 = configs_j[0]
    print(
        "Symbolic first call / XLA compile — often several minutes; please wait...",
        flush=True,
    )
    t0 = time.perf_counter()
    _block_until_ready(fd_sym(q0, v0, u0))
    print(f"  Symbolic compile done in {format_duration(time.perf_counter() - t0)}", flush=True)

print(f"Parity on {len(configs_j)} samples...", flush=True)
parity_check(fd_rnea, fd_aba, configs_j, "ABA")
if fd_sym is not None:
    parity_check(fd_rnea, fd_sym, configs_j, "Symbolic")
else:
    print("Symbolic plant missing — skip Symbolic parity.", flush=True)

print("Speed compare (warm-up + timed calls; per-call progress below)...", flush=True)
timing_all = speed_compare(named, configs_j, n_timing=N_TIMING, progress=True)
display(Markdown("### Speed (all pipelines)"))
display(Markdown(speed_table_markdown(timing_all)))


## 9. EoM integration — stepping $\dot x = f(x,u)$

Integrate catalog plants with fixed-step RK4. Symbolic stays out of the multi-step timer (closed-form $H$ is too slow per step).

| Phase | What it measures |
| --- | --- |
| **compile** | `Simulator` construction |
| **JIT warm-up** | first `solve()` |
| **integrate** | second `solve()` (EoM only) |

Grid: $t_f =$ `DEFAULT_INTEGRATION_TF`, $\Delta t =$ `DEFAULT_INTEGRATION_DT`, JAX.


In [ ]:
integ_result, ref_traj = eom_integration_comparison(
    params,
    symbolic_plant=None,  # single-step §8 already covers symbolic
    compile_backend=DEFAULT_INTEGRATION_BACKEND,
    tf=DEFAULT_INTEGRATION_TF,
    dt=DEFAULT_INTEGRATION_DT,
    progress=True,
)
print(
    f"EoM grid: tf={integ_result.tf}s, dt={integ_result.dt}s, "
    f"n={integ_result.n_steps}, backend={integ_result.backend}"
)

timing_cols = ["method", "compile", "jit_warmup", "integrate", "backend"]
display(Markdown("### EoM integration timings"))
display(
    Markdown(
        rows_to_markdown(
            integration_timing_rows(integ_result),
            timing_cols,
            duration_cols=("compile", "jit_warmup", "integrate"),
        )
    )
)
if integ_result.max_state_error:
    for name, err in integ_result.max_state_error.items():
        print(f"Trajectory parity {name} vs RNEA–H: max |Δx| = {format_error(err)}")


## 10. Summary

* **§6 catalog:** ABA matches RNEA–$H$; ABA is faster because it never forms $H$.
* **§7–8 symbolic:** same $\ddot q$ ballpark after a long derive/export; per-call speed is much slower (closed-form inertia).
* **§9 integration:** RNEA–$H$ and ABA trajectories agree on a short JAX grid; JIT warm-up dominates once, then integration is cheap.

**When to use which:** ABA for simulation loops; RNEA–$H$ / symbolic $H$ for teaching, linearization, and control design.
